# Prompt design case study — three wordings of the same AI-researcher prompt

The same two-track figure as
[two_track_figure.ipynb](two_track_figure.ipynb), over the prompt-design runs:
rows are prompt variants rather than ablations, and each is measured against the
original AI-researcher prompt in the strip on top. `*` marks significance at 95%
(corpus-level paired bootstrap, n=100,000).

Pairwise only — these runs have no pointwise data, and both tracks are the plan
form of the idea, which is what the AI-researcher prompt was written for.

Reads the `unified_*.json` chart data `merge_ablation_runs.py` writes; writes a
PDF (vector, for the paper) and a PNG to `output/figures/prompt-design/`. The
drawing is `common.TwoTrackFigure` — this notebook is only the wording.

In [ ]:
import sys
from pathlib import Path

from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src").is_dir() and (p / "output").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from novelty_eval.figures.common import TwoTrackFigure

print(ROOT)

## What the figure says

Everything that decides what this figure says is in the next cell. The two
tracks are separate merges, so a variant's row is the same wording scored
against a different baseline on each side.

In [ ]:
FIG = TwoTrackFigure(
    root=ROOT,
    figure="prompt-design",           # output subdirectory and filename prefix
    # (block title, merge dir under output/ablation_sweeps, name token)
    tracks=[("Human-only", "merge_ai_researcher_hvh-plan", "human"),
            ("Human + generated", "merge_ai_researcher_vanilla_ai-plan", "vanilla")],
    # (panel name, row label), top to bottom
    rows=[("ai_researcher_no_review", "No review mention"),
          ("ai_researcher_comparative", "Comparative language")],
    # judge columns, left to right
    models=["claude-opus-4-6", "gpt-5.4"],
    metrics={"pairwise": "pairwise_accuracy"},
    row_axis_label="Prompt variant",
    # The strip is not an ablated baseline here: it is the prompt as published.
    baseline_label="Original prompt",
)

## The numbers

`FIG.table(setup)` is the whole data path: one tidy row per (track, ablation,
judge), everything already in points. Print it, sort it, `.query()` it — if a
cell in the figure looks wrong, the reason is visible here, before any drawing
happens.

A missing panel, or a cell `common.UNSUPPORTED_CELLS` blanks (a judge that
cannot honour the ablation), gives `NaN` and draws as `—`.

In [ ]:
print(FIG.table(next(iter(FIG.metrics))).to_string())

## The figure: what each change did

One `pcolormesh` per track, then the cell text on top of it. Diverging red→green
scale centred on zero and shared across both blocks, so a colour means the same
thing left and right; every cell also prints its number, so colour is never the
only encoding. The baseline strip is drawn outside the grid, in a shade of its
own, so it never reads as a delta of zero.

In [ ]:
for setup in FIG.metrics:
    png = FIG.draw(setup)
    print(png.relative_to(ROOT), "(+ .pdf)")
    display(Image(filename=str(png)))

## The same numbers, absolute

The figure above colours the *change*, which hides where a judge started: two
judges that both drop 3 points look alike whether they land at 80 or at 55. This
is the same table and the same red→green scale, but a cell is the score itself
and nothing else, and the baseline is an ordinary row of the grid rather than a
strip measured against.

White is 50 — chance on a two-way decision — so red is a judge below it and
green one above, and the scale is symmetric about that rather than stretched to
whatever range this figure happens to cover. No significance markers: that is a
claim about the *change*, so it is made in the figure above. Written next to it
with an `_absolute` suffix.

In [ ]:
for setup in FIG.metrics:
    png = FIG.draw_absolute(setup)
    print(png.relative_to(ROOT), "(+ .pdf)")
    display(Image(filename=str(png)))